# AFK-Ultron YOLOv8 Two-Stage Training

**Before starting:** Go to `Runtime > Change runtime type` at the top menu and ensure your hardware accelerator is set to **T4 GPU**.

### 1. Install Dependencies

In [ ]:
!pip install ultralytics roboflow

### 2. Stage 1 - Aerial Pre-Training on VisDrone
This teaches the base YOLO network to recognize very small objects from a bird's eye perspective.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data="VisDrone.yaml", 
    epochs=50, 
    imgsz=1024, 
    batch=16,
    device=0, 
    name="aerial_base"
)

### 3. Stage 2 - Download SARD Dataset
Paste your Roboflow API key and Project ID here to download the Search & Rescue drone dataset.

In [ ]:
from roboflow import Roboflow

# ⚠️ UPDATE these values with your Roboflow SARD project details!
rf = Roboflow(api_key="YOUR_KEY_HERE")
project = rf.workspace("workspace-id").project("project-id") 
dataset = project.version(1).download("yolov8")

### 4. Stage 2 - Fine-Tuning
This applies heavy movement blur and camera tilts to focus specifically on humans.

In [ ]:
# Load weights from the Stage 1 run
model = YOLO('/content/runs/detect/aerial_base/weights/best.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml", 
    epochs=50, 
    imgsz=1024, 
    batch=16, 
    device=0, 
    name="sard_final",
    mosaic=1.0, 
    mixup=0.2, 
    degrees=15.0
)